# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Pregunta guía:** *¿Pueden convertir titulares peruanos en un clasificador de noticias?*

## Configuración inicial

| Parámetro | Valor |
| :--- | :--- |
| **Fecha de descarga** | 17 de octubre de 2026 (también queda en `FECHA_DESCARGA` al ejecutar) |
| **Opción elegida** | **C – Comparada**: La República + El Comercio (~200 noticias de cada medio) |
| **Categorías finales** | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |
| **URL base La República** | `https://larepublica.pe` |
| **URL base El Comercio** | `https://elcomercio.pe` |

### Fuentes

| Medio | Fuente | URLs |
| :--- | :--- | :--- |
| El Comercio | Sitemaps web por sección (últimas 100 notas de cada una) | `https://elcomercio.pe/sitemap/web/{politica, economia, deporte-total, tecnologia}/?outputType=xml` |
| La República | RSS oficiales | `https://larepublica.pe/rss/{politica, economia, deportes, ciencia, home}.xml` |
| La República | News sitemaps | `https://larepublica.pe/sitemap/{politica, economia, deportes, ciencia}.xml` |
| La República | Portadas de sección | `https://larepublica.pe/{politica, economia, deportes, ciencia, tecnologia}` |

### Estrategia de extracción (y por qué)

1. **El Comercio** publica un sitemap por sección con sus últimas 100 notas. Es la fuente más limpia: de cada
   sitemap se toma una muestra aleatoria (`SEED = 42`) y de cada nota se leen el titular (`<h1>`), la bajada
   (`og:description`) y la fecha (`article:published_time`).
2. **La República** solo expone en RSS y sitemaps las últimas ~48 horas (entre 7 y 25 notas por sección), y su
   `robots.txt` **prohíbe** la paginación de secciones (`/*/pag/*`, `/*/*/page/*`), el archivo (`/archivo/*`) y la
   API interna que usa el botón "cargar más" (`/data/*`). Por eso:
   - las fuentes oficiales se usan solo como **semillas**, y
   - se recorre el **grafo de enlaces relacionados**: cada nota enlaza a otras de su misma sección. Es un
     rastreo en anchura con una cola por categoría que se atiende en *round-robin* (el corpus queda balanceado)
     y que saca primero las notas más recientes (la fecha viene en la URL), para que ambos medios cubran una
     ventana temporal parecida.
3. **Buenas prácticas:** cada URL se valida contra `robots.txt` antes de pedirla, `User-Agent` descriptivo,
   pausa aleatoria de 1–2 s después de cada petición, máximo 1 reintento, log de todas las peticiones y
   respaldo crudo en `datos/raw/`.
4. **Reproducibilidad:** cada respuesta exitosa se guarda en una caché local (`datos/raw/html/`, fuera de Git
   por su tamaño). Si ya existe el respaldo crudo y `RE_SCRAPE = False`, el notebook se ejecuta completo sin
   volver a pedirle nada a los medios.

In [1]:
import gzip
import hashlib
import heapq
import json
import random
import re
import time
from collections import Counter
from datetime import date, datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import feedparser
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from protego import Protego

# Plotly interactivo en Jupyter + imagen PNG estática para que los gráficos se vean en GitHub
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width, pio.renderers["png"].height = 950, 500
pd.set_option("display.max_colwidth", 110)

FECHA_DESCARGA = date.today().isoformat()
SEED = 42
RE_SCRAPE = False            # True fuerza un scraping nuevo aunque exista el respaldo crudo
USER_AGENT = ("Tarea1-PythonIntermedio/1.0 (proyecto academico de clasificacion de titulares; "
              "scraping respetuoso; github.com/rodnm)")
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
META_POR_SECCION = 60        # notas por sección y medio: ~50 quedan tras la limpieza
MAX_ARTICULOS_LR = 500       # tope de seguridad del rastreo de La República

RAW = Path("datos/raw")
CACHE_HTML = RAW / "html"
CACHE_HTML.mkdir(parents=True, exist_ok=True)
RUTA_CRUDO = RAW / "articulos_crudos.csv"
RUTA_LOG = RAW / "log_peticiones.csv"
RUTA_NO_NOTAS = RAW / "urls_no_nota.csv"

random.seed(SEED)
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
print("Fecha de descarga:", FECHA_DESCARGA)

Fecha de descarga: 2026-10-17


## 1. Cliente HTTP respetuoso

`urllib.robotparser`, de la librería estándar, **no entiende comodines** (`*`, `$`), y La República los usa
(`/*/pag/*`). Con ese parser, `/politica/pag/2` aparecería como permitido. Por eso se usa `protego`, el intérprete
de `robots.txt` de Scrapy, que sí implementa los comodines.

La función `obtener()` concentra todas las reglas: valida `robots.txt`, hace una pausa de 1–2 s después de cada
petición, reintenta como máximo una vez (y nunca ante un 403/404/410, que son definitivos), registra cada petición
en el log y guarda en caché solo las respuestas 200. Una página en caché es, entonces, una URL que funcionó.

In [2]:
ROBOTS = {}
LOG = []


def reglas_robots(dominio):
    """Lee robots.txt una vez por dominio y guarda una copia cruda."""
    if dominio not in ROBOTS:
        ruta = RAW / f"robots_{dominio}.txt"
        if ruta.exists() and not RE_SCRAPE:
            texto = ruta.read_text(encoding="utf-8")
        else:
            texto = session.get(f"https://{dominio}/robots.txt", timeout=20).text
            ruta.write_text(texto, encoding="utf-8")
            time.sleep(random.uniform(1, 2))
        ROBOTS[dominio] = Protego.parse(texto)
    return ROBOTS[dominio]


def ruta_cache(url):
    return CACHE_HTML / f"{hashlib.sha1(url.encode()).hexdigest()}.html.gz"


def obtener(url):
    """GET respetuoso. Devuelve el texto de la respuesta o None si no se pudo obtener."""
    cache = ruta_cache(url)
    if cache.exists() and not RE_SCRAPE:
        return gzip.decompress(cache.read_bytes()).decode("utf-8")
    if not reglas_robots(urlparse(url).netloc).can_fetch(url, USER_AGENT):
        LOG.append({"url": url, "estado": "bloqueado_robots", "intento": 0,
                    "hora": datetime.now().isoformat(timespec="milliseconds")})
        return None
    for intento in (1, 2):                       # 1 intento + máximo 1 reintento
        try:
            r = session.get(url, timeout=25)
            estado = r.status_code
        except requests.RequestException as e:
            r, estado = None, f"error:{type(e).__name__}"
        LOG.append({"url": url, "estado": estado, "intento": intento,
                    "hora": datetime.now().isoformat(timespec="milliseconds")})
        time.sleep(random.uniform(1, 2))
        if estado == 200:
            cache.write_bytes(gzip.compress(r.content))
            return r.content.decode("utf-8", errors="replace")   # ambos medios sirven UTF-8
        if estado in (403, 404, 410):            # errores definitivos: no se reintenta
            return None
    return None


pruebas = [
    "https://larepublica.pe/rss/politica.xml",
    "https://larepublica.pe/economia/2026/10/16/ejemplo-123",
    "https://larepublica.pe/politica/pag/2",
    "https://larepublica.pe/data/noticias",
    "https://larepublica.pe/archivo/2026",
    "https://elcomercio.pe/sitemap/web/politica/?outputType=xml",
    "https://elcomercio.pe/buscar/congreso/",
]
display(pd.DataFrame({
    "url": pruebas,
    "permitido": [reglas_robots(urlparse(u).netloc).can_fetch(u, USER_AGENT) for u in pruebas],
}))

,url,permitido
0,https://larepublica.pe/rss/politica.xml,True
1,https://larepublica.pe/economia/2026/10/16/ejemplo-123,True
2,https://larepublica.pe/politica/pag/2,False
3,https://larepublica.pe/data/noticias,False
4,https://larepublica.pe/archivo/2026,False
5,https://elcomercio.pe/sitemap/web/politica/?outputType=xml,True
6,https://elcomercio.pe/buscar/congreso/,False


## 2. Funciones de extracción

* **`seccion_original`** se toma de la ruta de la URL, que es como cada diario clasifica la nota en su web:
  `elcomercio.pe/deporte-total/futbol-peruano/...-noticia/` → `deporte-total/futbol-peruano` y
  `larepublica.pe/economia/2026/10/16/...` → `economia`.
* **Titular:** `<h1>` (lo que ve el lector); si falta, `og:title`. **Bajada:** `og:description`.
  **Fecha:** `article:published_time` (El Comercio) o `datePublished` del JSON-LD (La República, que no publica
  la etiqueta `meta`).

In [3]:
PATRON_LR = re.compile(r"^https://larepublica\.pe/(?P<seccion>[a-z0-9-]+(?:/[a-z0-9-]+)*)/"
                       r"(?P<fecha>20\d{2}/\d{2}/\d{2})/[a-z0-9-]+$")
PATRON_EC = re.compile(r"^https://elcomercio\.pe/(?P<seccion>[a-z0-9-]+(?:/[a-z0-9-]+)*)/"
                       r"[a-z0-9-]+-noticia/?$")
SUFIJO_MEDIO = re.compile(r"\s*\|\s*(El Comercio( Perú)?|La República)\s*$", re.IGNORECASE)


def normalizar_lr(url):
    """URL canónica de La República: absoluta, sin parámetros, sin /amp/ ni barra final."""
    url = urljoin("https://larepublica.pe", url.strip()).split("#")[0].split("?")[0].rstrip("/")
    return url.replace("https://larepublica.pe/amp/", "https://larepublica.pe/")


def seccion_de_url(url):
    m = PATRON_LR.match(url) or PATRON_EC.match(url)
    return m["seccion"] if m else None


def fecha_json_ld(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            datos = json.loads(script.string or "")
        except json.JSONDecodeError:
            continue
        nodos = datos if isinstance(datos, list) else datos.get("@graph", [datos])
        for nodo in nodos:
            if isinstance(nodo, dict) and nodo.get("datePublished"):
                return nodo["datePublished"]
    return None


def parsear_articulo(html, url, medio):
    soup = BeautifulSoup(html, "lxml")

    def meta(clave):
        tag = soup.find("meta", property=clave) or soup.find("meta", attrs={"name": clave})
        return tag["content"].strip() if tag and tag.get("content") else None

    h1 = soup.find("h1")
    titular = h1.get_text(" ", strip=True) if h1 else (meta("og:title") or "")
    fecha = meta("article:published_time") or fecha_json_ld(soup)
    return {
        "titular": SUFIJO_MEDIO.sub("", titular),
        "bajada": meta("og:description") or meta("description") or "",
        "medio": medio,
        "seccion_original": seccion_de_url(url),
        "url": url,
        "fecha": fecha[:10] if fecha else None,
    }

## 3. El Comercio: sitemaps por sección

Del sitemap de cada sección se descartan antes de pedirlas las URLs que no son notas (videos o galerías, que no
terminan en `-noticia/`) y se toma una muestra aleatoria de `META_POR_SECCION` notas.

In [4]:
EC_SECCIONES = ["politica", "economia", "deporte-total", "tecnologia"]


def recolectar_ec():
    registros, no_notas = [], []
    for seccion in EC_SECCIONES:
        xml = obtener(f"https://elcomercio.pe/sitemap/web/{seccion}/?outputType=xml")
        if not xml:
            print(f"  sitemap {seccion}: sin respuesta")
            continue
        (RAW / f"ec_sitemap_{seccion}.xml").write_text(xml, encoding="utf-8")
        locs = re.findall(r"<loc>\s*([^<\s]+)\s*</loc>", xml)
        notas = [u for u in locs if PATRON_EC.match(u)]
        no_notas += [{"medio": "El Comercio", "url": u} for u in locs if not PATRON_EC.match(u)]
        muestra = random.Random(SEED).sample(notas, min(META_POR_SECCION, len(notas)))
        print(f"  sitemap {seccion:<14}: {len(locs)} URLs, {len(notas)} notas, muestra de {len(muestra)}")
        for url in muestra:
            html = obtener(url)
            if html:
                registros.append({**parsear_articulo(html, url, "El Comercio"),
                                  "fuente": f"sitemap_ec:{seccion}"})
    return registros, no_notas

## 4. La República: semillas + rastreo de enlaces relacionados

* **Semillas:** RSS, news sitemaps y portadas de sección. El RSS `home.xml` mezcla todas las secciones del diario;
  sus notas de otras secciones (mundo, sociedad, espectáculos…) se guardan tal cual para **documentar en la
  limpieza las noticias que no encajan** en las 4 categorías.
* **Rastreo:** cada categoría tiene su propia cola de prioridad (un *heap* ordenado por la fecha de la URL, la
  más reciente primero). En cada vuelta se atiende una nota de cada categoría que todavía no llegó a la meta,
  y los enlaces a notas de las secciones objetivo que aparecen en ella se agregan a la cola correspondiente.

In [5]:
LR_SECCION_A_CATEGORIA = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}


def categoria_rastreo(url):
    seccion = seccion_de_url(url)
    return LR_SECCION_A_CATEGORIA.get(seccion.split("/")[0]) if seccion else None


def enlaces_lr(html):
    soup = BeautifulSoup(html, "lxml")
    urls = {normalizar_lr(a["href"]) for a in soup.find_all("a", href=True)}
    return {u for u in urls if PATRON_LR.match(u)}


def semillas_lr():
    urls, otras_secciones = set(), []
    for nombre in ["politica", "economia", "deportes", "ciencia", "home"]:
        xml = obtener(f"https://larepublica.pe/rss/{nombre}.xml")
        if not xml:
            continue
        (RAW / f"lr_rss_{nombre}.xml").write_text(xml, encoding="utf-8")
        for item in feedparser.parse(xml).entries:
            url = normalizar_lr(item.link)
            if categoria_rastreo(url):
                urls.add(url)
            elif PATRON_LR.match(url):
                publicada = item.get("published_parsed")
                otras_secciones.append({
                    "titular": item.title.strip(), "bajada": "", "medio": "La Republica",
                    "seccion_original": seccion_de_url(url), "url": url,
                    "fecha": time.strftime("%Y-%m-%d", publicada) if publicada else None,
                    "fuente": f"rss_lr:{nombre}",
                })
    for nombre in ["politica", "economia", "deportes", "ciencia"]:
        xml = obtener(f"https://larepublica.pe/sitemap/{nombre}.xml")
        if xml:
            (RAW / f"lr_sitemap_{nombre}.xml").write_text(xml, encoding="utf-8")
            locs = (normalizar_lr(u) for u in re.findall(r"<loc>\s*([^<\s]+)\s*</loc>", xml))
            urls.update(u for u in locs if categoria_rastreo(u))
    for nombre in ["politica", "economia", "deportes", "ciencia", "tecnologia"]:
        html = obtener(f"https://larepublica.pe/{nombre}")
        if html:
            urls.update(u for u in enlaces_lr(html) if categoria_rastreo(u))
    return urls, otras_secciones


def rastrear_lr(semillas):
    colas = {c: [] for c in CATEGORIAS}
    vistos, registros, conteo = set(), [], Counter()

    def encolar(url):
        categoria = categoria_rastreo(url)
        if categoria and url not in vistos:
            vistos.add(url)
            fecha = int(PATRON_LR.match(url)["fecha"].replace("/", ""))
            heapq.heappush(colas[categoria], (-fecha, url))   # fecha negativa: la más reciente sale primero

    for url in sorted(semillas):
        encolar(url)
    print("Cola inicial por categoría:", {c: len(q) for c, q in colas.items()})

    procesados, inicio = 0, time.time()
    while procesados < MAX_ARTICULOS_LR:
        pendientes = [c for c in CATEGORIAS if conteo[c] < META_POR_SECCION and colas[c]]
        if not pendientes:
            break
        for categoria in pendientes:             # round-robin: una nota por categoría en cada vuelta
            _, url = heapq.heappop(colas[categoria])
            html = obtener(url)
            procesados += 1
            if not html:
                continue
            registros.append({**parsear_articulo(html, url, "La Republica"), "fuente": "rastreo_lr"})
            conteo[categoria] += 1
            for enlace in enlaces_lr(html):
                encolar(enlace)
    print(f"Rastreo terminado: {procesados} notas procesadas en {time.time() - inicio:.0f} s → {dict(conteo)}")
    return registros

## 5. Ejecución del scraping (o carga del respaldo crudo)

Si `datos/raw/articulos_crudos.csv` ya existe y `RE_SCRAPE = False`, se carga el respaldo en lugar de volver a
descargar. El log de peticiones se conserva para documentar cuándo se hizo el scraping y con qué pausas.

In [6]:
if RUTA_CRUDO.exists() and not RE_SCRAPE:
    crudo = pd.read_csv(RUTA_CRUDO)
    no_notas = pd.read_csv(RUTA_NO_NOTAS)
    log = pd.read_csv(RUTA_LOG)
    print(f"Respaldo crudo cargado: {len(crudo)} registros (RE_SCRAPE = False).")
else:
    print("El Comercio:")
    registros_ec, urls_no_nota = recolectar_ec()
    print("La República:")
    semillas, otras_secciones_lr = semillas_lr()
    print(f"  URLs semilla en las secciones objetivo: {len(semillas)}")
    registros_lr = rastrear_lr(semillas)

    crudo = pd.DataFrame(registros_ec + registros_lr + otras_secciones_lr)
    no_notas = pd.DataFrame(urls_no_nota, columns=["medio", "url"])
    log = pd.DataFrame(LOG)
    crudo.to_csv(RUTA_CRUDO, index=False, encoding="utf-8")
    no_notas.to_csv(RUTA_NO_NOTAS, index=False, encoding="utf-8")
    log.to_csv(RUTA_LOG, index=False, encoding="utf-8")
    print(f"Respaldo crudo guardado: {len(crudo)} registros en {RUTA_CRUDO}")

horas = pd.to_datetime(log["hora"])
print(f"\nScraping: {len(log)} peticiones HTTP entre {horas.min():%Y-%m-%d %H:%M} y {horas.max():%H:%M} "
      f"| pausa mediana entre peticiones: {horas.diff().dt.total_seconds().median():.1f} s")
display(log["estado"].astype(str).value_counts().rename_axis("estado HTTP").to_frame("peticiones"))
display(crudo.groupby(["medio", "fuente"]).size().rename("registros").to_frame())

El Comercio:


  sitemap politica      : 100 URLs, 100 notas, muestra de 60


  sitemap economia      : 100 URLs, 100 notas, muestra de 60


  sitemap deporte-total : 100 URLs, 100 notas, muestra de 60


  sitemap tecnologia    : 100 URLs, 97 notas, muestra de 60


La República:


  URLs semilla en las secciones objetivo: 76
Cola inicial por categoría: {'politica': 26, 'economia': 14, 'deportes': 14, 'ciencia_tecnologia': 22}


Rastreo terminado: 240 notas procesadas en 424 s → {'politica': 60, 'economia': 60, 'deportes': 60, 'ciencia_tecnologia': 60}
Respaldo crudo guardado: 627 registros en datos\raw\articulos_crudos.csv

Scraping: 498 peticiones HTTP entre 2026-10-17 10:49 y 11:08 | pausa mediana entre peticiones: 2.1 s


,peticiones
estado HTTP,
200,498


registros
medio        fuente                             
El Comercio  sitemap_ec:deporte-total         60
             sitemap_ec:economia              60
             sitemap_ec:politica              60
             sitemap_ec:tecnologia            60
La Republica rastreo_lr                      240
             rss_lr:home                     147